# Python pyannote and whisper by hand


In [1]:
# setup python system path if needed
import sys, json
from pathlib import Path

parent_dir = Path.cwd().resolve().parent # parent = project root
ffmpeg_bin_path = parent_dir / "ffmpeg" / "bin"
sys.path.append(str(ffmpeg_bin_path))

def print_formatted_sys_path():
    # Rather than deal with the raw output of sys.path, we can use the json module to spit out a nicely formatted object
    formatted_path = json.dumps(sys.path, indent=4)

    # use some colors to make it pretty!
    print("\033[1;34m[SYS.PATH]\033[0m")  # ANSI code for blue for the title
    print("\033[1;32m" + formatted_path + "\033[0m")  # ANSI code for green for paths, then ANSI code for reset


# Call the function to print the formatted sys.path
print_formatted_sys_path()


[SYS.PATH]
[
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.14.2-windows-x86_64-none\\python314.zip",
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.14.2-windows-x86_64-none\\DLLs",
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.14.2-windows-x86_64-none\\Lib",
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.14.2-windows-x86_64-none",
    "z:\\code\\STAT405_AudioNotetaker\\.venv",
    "",
    "z:\\code\\STAT405_AudioNotetaker\\.venv\\Lib\\site-packages",
    "\\\\zdrive.labs.cset.oit.edu\\zdrive\\andrew.sparkes\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin"
]


In [2]:
# setup stuff

import log_config # to override default and use loguru instead
log_config.setup_logging()

from loguru import logger

from pydantic_settings import BaseSettings, SettingsConfigDict

class NotebookSettings(BaseSettings):
    hf_token: str
    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8", extra="allow")

settings = NotebookSettings()

#print(f"HF_TOKEN env variable: {settings.hf_token}")


In [3]:
# user configuration settings
audio_file = Path("sample_data/en_US/Gene_Hackman_Interview_by_Bob_Lardine.mp4")
output_dir = Path("sample_output")

HF_token = settings.hf_token


In [5]:



from pyannote.audio import Pipeline
import whisper

# 1. Load Diarization Pipeline
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
diarization = pipeline(str(audio_file))

# 2. Transcribe with Whisper
model = whisper.load_model("small")
result = model.transcribe(str(audio_file))

# 3. Map speaker labels to transcription segments
for segment, _, speaker in diarization.itertracks(yield_label=True):
    print(f"{segment.start:.1f}s - {segment.end:.1f}s: {speaker}")
    # Match segments with whisper result timestamps


z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             tab

TypeError: LoadLibrary() argument 1 must be str, not None